PRE-REQUISITES
- runnable()
- is_instance()

### STREAMING

Possible things with streaming:
1. stream agent process
2. stream llm tokens
3. stream reasoning/thinking steps
4. stream custom updates
5. mulitple modes of streaming


#### <u>stream modes:</u>


1. updates:
    - Streams state updates after each agent step. 
    - If multiple updates are made in the same step, those updates are streamed separately.

2. messages:
    - Streams tuples of (token, metadata) from any graph nodes where an LLM is invoked.
3. custom:
    - Streams custom data from inside your graph nodes using the stream writer.

COMMON PATTERNS
1. Streaming thinking tokens:
    - to stream thinking token from agent use --> stream_mode = 'messages'
    - and  filtering standard content blocks for the type "reasoning"

2. Streaming tool calls:
    - stream_mode="messages" --> will stream incremental message chunks generated by all LLM calls in the agent.
    - To access the completed messages with parsed tool calls.
        1. If those messages are tracked in the state :=> stream_mode=["messages", "updates"]
        2. If those messages are not tracked in the state, use custom updates or aggregate the chunks during the streaming loop

3. Streaming human in the loop:



##### updates

```python
from langchain.agents import create_agent


def get_weather(city: str) -> str:
    """Get weather for a given city."""

    return f"It's always sunny in {city}!"

agent = create_agent(
    model="gpt-5-nano",
    tools=[get_weather],
)
for chunk in agent.stream(
    {"messages": [{"role": "user", "content": "What is the weather in SF?"}]},
    stream_mode="updates",
    version="v2",
):
    if chunk["type"] == "updates":
        for step, data in chunk["data"].items():
            print(f"step: {step}")
            print(f"content: {data['messages'][-1].content_blocks}")

##### Updates and custom

```python
from langchain.agents import create_agent
from langgraph.config import get_stream_writer


def get_weather(city: str) -> str:
    """Get weather for a given city."""
    writer = get_stream_writer()
    writer(f"Looking up data for city: {city}")
    writer(f"Acquired data for city: {city}")
    return f"It's always sunny in {city}!"

agent = create_agent(
    model="gpt-5-nano",
    tools=[get_weather],
)

for chunk in agent.stream(
    {"messages": [{"role": "user", "content": "What is the weather in SF?"}]},
    stream_mode=["updates", "custom"],
    version="v2",
):
    print(f"stream_mode: {chunk['type']}")
    print(f"content: {chunk['data']}")
    print("\n")

##### Streaming thinking / reasoning tokens

``` python
from langchain.agents import create_agent
from langchain.messages import AIMessageChunk
from langchain_anthropic import ChatAnthropic
from langchain_core.runnables import Runnable


def get_weather(city: str) -> str:
    """Get weather for a given city."""
    return f"It's always sunny in {city}!"


model = ChatAnthropic(
    model_name="claude-sonnet-4-6",
    timeout=None,
    stop=None,
    thinking={"type": "enabled", "budget_tokens": 5000},
)
agent: Runnable = create_agent(
    model=model,
    tools=[get_weather],
)

for token, metadata in agent.stream(
    {"messages": [{"role": "user", "content": "What is the weather in SF?"}]},
    stream_mode="messages",
):
    if not isinstance(token, AIMessageChunk):
        continue
    reasoning = [b for b in token.content_blocks if b["type"] == "reasoning"]
    text = [b for b in token.content_blocks if b["type"] == "text"]
    if reasoning:
        print(f"[thinking] {reasoning[0]['reasoning']}", end="")
    if text:
        print(text[0]["text"], end="")

##### Streaming tool calls